In [ ]:
import json
import os

import pandas as pd
import requests
from pyspark.sql.functions import expr

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))))
config_dir = os.path.join(repo_root, "config", "data_ingest")

with open(os.path.join(config_dir, "catalog_config.json")) as f:
    catalog = json.load(f)["catalog"]

fred_config_path = os.path.join(config_dir, "fred_config.json")
with open(fred_config_path) as f:
    fred_config = json.load(f)

series_list = fred_config["series"]
fred_api_key = dbutils.secrets.get(scope="finsight", key="fred-api-key")

In [ ]:
def fetch_observations(series):
    params = {
        "series_id": series["series_id"],
        "api_key": fred_api_key,
        "file_type": "json",
    }
    if series["start_date"]:
        params["observation_start"] = series["start_date"]

    response = requests.get("https://api.stlouisfed.org/fred/series/observations", params=params)
    response.raise_for_status()
    observations = response.json()["observations"]

    df = pd.DataFrame(observations)[["date", "value"]]
    df = df[df["value"] != "."]
    df["value"] = df["value"].astype(float)
    return df.rename(columns={"value": series["series_id"]})

series_frames = {series["series_id"]: fetch_observations(series) for series in series_list}

In [ ]:
indicators_pd = series_frames[series_list[0]["series_id"]]
for series_id in list(series_frames)[1:]:
    indicators_pd = indicators_pd.merge(series_frames[series_id], on="date", how="outer")
indicators_pd["date"] = pd.to_datetime(indicators_pd["date"]).dt.date

indicators = spark.createDataFrame(indicators_pd)

dqr = spark.table(f"{catalog}.config.data_quality_rules")\
    .filter("domain = 'fred'")\
    .select("type","column","rule")\
    .collect()

for r in dqr:
    if r.type == "case":
        indicators = indicators.withColumn(r.column, expr(r.rule))


indicators.createOrReplaceTempView("fred_staging")

In [ ]:
series_ids = [series["series_id"] for series in series_list]
columns_ddl = ",\n".join(f"{sid} DOUBLE" for sid in series_ids)
update_set = ",\n".join(f"{sid} = COALESCE(source.{sid}, target.{sid})" for sid in series_ids)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.fred")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.fred.macro_indicators (
        date DATE,
        {columns_ddl}
    )
    USING DELTA
""")

spark.sql(f"""
    MERGE INTO {catalog}.fred.macro_indicators AS target
    USING fred_staging AS source
    ON target.date = source.date
    WHEN MATCHED THEN UPDATE SET {update_set}
    WHEN NOT MATCHED THEN INSERT *
""")

In [ ]:
for series in series_list:
    max_date = series_frames[series["series_id"]]["date"].max()
    if pd.notna(max_date):
        series["start_date"] = max_date

with open(fred_config_path, "w") as f:
    json.dump(fred_config, f, indent=4)